# GRPO Fine-tuning (Advanced) — Chatbot Tim Legal berbasis RAG

**Peserta**: Nazhif Setya Nugroho
**Program**: Dicoding — Pengembangan Generative AI Berbasis LLM (PGABL)
**Target level**: Advanced ⭐⭐⭐⭐⭐

## Overview

Notebook ini melanjutkan model hasil **SFT** (`PGABL-Llama-3.2-3B-SFT`) dengan **GRPO (Group Relative Policy Optimization)** untuk mengajarkan model **reasoning terstruktur** di dalam tag `<think>...</think>` sebelum menjawab, sambil menjaga jawaban tetap dalam **Bahasa Indonesia** formal.

GRPO mengoptimasi kebijakan (policy) model terhadap **4 reward function** yang mengukur kualitas keluaran secara relatif antar kelompok generasi (group-relative), tanpa perlu model reward terpisah:

| Reward | Bobot | Fungsi |
|---|---|---|
| `format_reward_func` | 0.30 | Kehadiran blok `<think>...</think>` |
| `reasoning_length_reward` | 0.15 | Panjang reasoning ideal (kurva lonceng, puncak ~300 char) |
| `correctness_reward` | 0.40 | ROUGE-L jawaban final vs ground-truth |
| `language_reward_func` | 0.15 | Reward Bahasa Indonesia, penalti dominan Inggris |

## Desain kunci

- **Basis**: melanjutkan dari model SFT winner (bukan base) — GRPO adalah tahap alignment di atas model yang sudah instruction-tuned.
- **Dataset**: subset `Ichsan2895/alpaca-gpt4-indonesian` (Q&A generik Bahasa Indonesia). Pengetahuan hukum spesifik **tidak** dihafal ke bobot — itu disuntikkan via RAG saat inference. GRPO di sini melatih *cara berpikir dan berbahasa*, bukan substansi hukum.
- **Test case wajib**: `"Berapa hak lembur staf admin menurut PP 35/2021?"` → output harus menampilkan reasoning `<think>...</think>`.
- **Output**: model di-push sebagai `merged_16bit` ke Hugging Face Hub (public).

## Prasyarat runtime

1. Colab **T4 GPU** (16 GB VRAM).
2. Colab Secrets: `HF_TOKEN` (scope Write), `WANDB_API_KEY`.
3. Model SFT sudah tersedia di Hub: `PGABL-Llama-3.2-3B-SFT` (hasil notebook Fine-tuning).
4. Google Drive mounted untuk checkpoint (resume-safe lintas sesi).

**Estimasi Run All**: ~60-120 menit (GRPO 300 steps + merge + upload). Resume dari checkpoint jauh lebih cepat.

---

## Section 1 — Setup Environment

In [1]:
import os, sys, json, gc, random, time, subprocess
from pathlib import Path

assert 'google.colab' in sys.modules, 'Notebook ini didesain untuk Google Colab T4.'

from google.colab import userdata, drive

HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
assert HF_TOKEN, 'HF_TOKEN tidak ditemukan di Colab Secrets.'
assert WANDB_API_KEY, 'WANDB_API_KEY tidak ditemukan di Colab Secrets.'

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_PROJECT'] = 'PGABL-GRPO'

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PGABL')
GRPO_CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints' / 'grpo'
EVIDENCE_DIR = DRIVE_ROOT / 'outputs' / 'finetune_evidence'
for d in (GRPO_CHECKPOINTS_DIR, EVIDENCE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)

print(f'HF_TOKEN loaded (starts with {HF_TOKEN[:6]}...)')
print(f'GRPO checkpoints dir: {GRPO_CHECKPOINTS_DIR}')
print(f'Seed: {SEED}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_TOKEN loaded (starts with hf_bRo...)
GRPO checkpoints dir: /content/drive/MyDrive/PGABL/checkpoints/grpo
Seed: 42


In [2]:
# Defensive install (jika notebook di-run standalone)
import importlib

required_packages = ['unsloth', 'trl', 'transformers', 'datasets', 'peft', 'wandb', 'rouge_score', 'langdetect']
missing = [p for p in required_packages if importlib.util.find_spec(p) is None]

if missing:
    print(f'Installing missing packages: {missing}')
    !pip install -q "unsloth[colab-new]"
    !pip install -q wandb rouge-score langdetect
    print('\n' + '='*60)
    print('INSTALL DONE. Runtime perlu di-restart supaya versi baru ter-load.')
    print('Menu: Runtime -> Restart runtime, lalu Run All lagi.')
    print('Cell ini akan otomatis skip install saat re-run.')
    print('='*60)
    # Force kernel restart (Colab akan auto-reconnect)
    os.kill(os.getpid(), 9)

print('All required packages present.')

All required packages present.


In [3]:
import torch
import numpy as np
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
# Unsloth versi baru auto-patch GRPO saat import; guard untuk versi lama
try:
    from unsloth import PatchFastRL
    PatchFastRL('GRPO', FastLanguageModel)
except Exception:
    pass
from trl import GRPOConfig, GRPOTrainer
from transformers import set_seed
import wandb

set_seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), 'CUDA tidak tersedia. Runtime → Change runtime type → T4 GPU.'

print(f'torch: {torch.__version__}')
print(f'CUDA device: {torch.cuda.get_device_name(0)}')
!nvidia-smi --query-gpu=memory.total,memory.free --format=csv,noheader

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
torch: 2.10.0+cu128
CUDA device: Tesla T4
15360 MiB, 14758 MiB


In [4]:
# Login HuggingFace + Weights & Biases
from huggingface_hub import login as hf_login, HfApi

hf_login(token=HF_TOKEN, add_to_git_credential=False)
hf_api = HfApi()
HF_USERNAME = hf_api.whoami()['name']
print(f'HuggingFace login OK sebagai: {HF_USERNAME}')

wandb.login(key=WANDB_API_KEY)
print('Weights & Biases login OK')

HuggingFace login OK sebagai: nazhifsetya-merpati


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nazhif-sn (nazhif-sn-merpati) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Weights & Biases login OK


---

## Section 2 — Load Model SFT + LoRA GRPO

GRPO melanjutkan dari **model SFT winner** (`PGABL-Llama-3.2-3B-SFT`), di-load 4-bit lalu ditempeli **LoRA adapter baru** untuk fase RL. LoRA menyasar MHA + FFN (7 modul) — sama seperti SFT.

In [5]:
SFT_MODEL = f'{HF_USERNAME}/PGABL-Llama-3.2-3B-SFT'   # repo = LoRA adapter hasil SFT
MAX_SEQ_LENGTH = 1024                                  # cukup untuk prompt(512)+completion(384)

# Repo SFT berisi LoRA adapter (base = unsloth/Llama-3.2-3B-Instruct). Unsloth memuat
# base + adapter sekaligus, siap DILANJUTKAN training. Karena adapter sudah menempel,
# kita TIDAK memanggil get_peft_model lagi (itu memicu "model already has LoRA adapters").
print(f'Loading SFT adapter {SFT_MODEL} (base + LoRA, 4-bit)...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
)

# Pastikan adapter dalam mode trainable untuk lanjutan GRPO (bukan inference).
try:
    FastLanguageModel.for_training(model)
except Exception:
    pass

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)')
assert trainable > 0, 'Adapter SFT ter-load non-trainable — cek pemuatan (butuh for_training).'


Loading SFT adapter nazhifsetya-merpati/PGABL-Llama-3.2-3B-SFT (base + LoRA, 4-bit)...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Trainable params: 48,627,712 / 3,261,377,536 (1.4910%)


---

## Section 3 — Build Dataset GRPO

GRPO butuh kolom `prompt` (conversational) + kolom ground-truth untuk `correctness_reward`. Kita ambil subset `Ichsan2895/alpaca-gpt4-indonesian`, bungkus tiap pertanyaan dengan **system prompt** yang meminta model berpikir di dalam `<think>...</think>` lalu menjawab dalam Bahasa Indonesia.

In [6]:
SYSTEM_PROMPT = (
    'Anda adalah asisten hukum. Selalu berpikir dulu di dalam <think>...</think> '
    'sebelum memberikan jawaban akhir dalam Bahasa Indonesia.'
)

GRPO_SUBSET_SIZE = 2000   # 300 steps x 4 prompt/step = 1200 prompt; 2000 beri variasi

raw = load_dataset('Ichsan2895/alpaca-gpt4-indonesian', split='train')
print(f'Dataset penuh: {len(raw):,} rows | kolom: {raw.column_names}')

def _valid(row):
    q, a = (row.get('input') or '').strip(), (row.get('output') or '').strip()
    return 8 <= len(q) <= 400 and len(a) >= 8   # buang terlalu pendek/panjang

filtered = raw.filter(_valid)
subset = filtered.shuffle(seed=SEED).select(range(min(GRPO_SUBSET_SIZE, len(filtered))))

def to_grpo(row):
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row['input'].strip()},
        ],
        'ground_truth': row['output'].strip(),
    }

grpo_ds = subset.map(to_grpo, remove_columns=subset.column_names)
print(f'GRPO dataset siap: {len(grpo_ds):,} rows')
print('\nContoh 1 prompt:')
print(json.dumps(grpo_ds[0]['prompt'], ensure_ascii=False, indent=2))
print('ground_truth:', grpo_ds[0]['ground_truth'][:160], '...')

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Dataset penuh: 49,969 rows | kolom: ['Unnamed: 0', 'input', 'output']


Filter:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

GRPO dataset siap: 2,000 rows

Contoh 1 prompt:
[
  {
    "role": "system",
    "content": "Anda adalah asisten hukum. Selalu berpikir dulu di dalam <think>...</think> sebelum memberikan jawaban akhir dalam Bahasa Indonesia."
  },
  {
    "role": "user",
    "content": "Jelaskan mengapa air mendidih membeku lebih cepat daripada air dingin"
  }
]
ground_truth: Fenomena ini dikenal sebagai efek Mpemba, dinamai dari mahasiswa Tanzania Erasto Mpemba yang pertama kali mengamatinya pada tahun 1963. Meskipun efek ini telah  ...


---

## Section 4 — 4 Reward Function (Rubric WAJIB Advanced)

Empat reward function inti GRPO. Signature mengikuti TRL GRPOTrainer: `f(prompts, completions, **kwargs) -> list[float]`; kolom dataset (`ground_truth`) diterima sebagai kwargs. Skor **mentah** dikembalikan tiap fungsi; bobot antar-reward di-apply oleh `GRPOConfig.reward_weights`.

In [7]:
import re, math

THINK_PATTERN = re.compile(r'<think>(.*?)</think>', re.DOTALL)
REASONING_PEAK_CHAR = 300
REASONING_SPREAD = 150
LANG_ID_THRESHOLD = 0.7
LANG_EN_PENALTY_THRESHOLD = 0.3
LANG_PENALTY_VALUE = -0.5


def _completion_text(completion):
    # support conversational [{'role','content'}] dan string
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion:
        last = completion[-1]
        return last.get('content', '') if isinstance(last, dict) else str(last)
    if isinstance(completion, dict):
        return completion.get('content', '')
    return str(completion)


def _extract_think(text):
    m = THINK_PATTERN.search(text)
    return m.group(1).strip() if m else ''


def _extract_answer(text):
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


# ---------- 1. format ----------
def format_reward_func(prompts=None, completions=None, **kwargs):
    return [1.0 if THINK_PATTERN.search(_completion_text(c)) else 0.0 for c in completions]


# ---------- 2. reasoning length ----------
def reasoning_length_reward(prompts=None, completions=None, **kwargs):
    out = []
    for c in completions:
        think = _extract_think(_completion_text(c))
        if not think:
            out.append(0.0); continue
        r = 1.0 / (1.0 + ((len(think) - REASONING_PEAK_CHAR) / REASONING_SPREAD) ** 2)
        out.append(float(r))
    return out


# ---------- 3. correctness (ROUGE-L) ----------
from rouge_score import rouge_scorer
_SCORER = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def _rouge_l(pred, ref):
    if not pred or not ref:
        return 0.0
    return float(_SCORER.score(ref, pred)['rougeL'].fmeasure)

def correctness_reward(prompts=None, completions=None, ground_truth=None, **kwargs):
    texts = [_completion_text(c) for c in completions]
    if ground_truth is None:
        ground_truth = [''] * len(texts)
    return [_rouge_l(_extract_answer(t).lower(), (gt or '').lower())
            for t, gt in zip(texts, ground_truth)]


# ---------- 4. language ----------
from langdetect import detect_langs, DetectorFactory
DetectorFactory.seed = 42

def _id_prob(text):
    if not text.strip():
        return 0.0
    try:
        return {l.lang: l.prob for l in detect_langs(text)}.get('id', 0.0)
    except Exception:
        return 0.0

def language_reward_func(prompts=None, completions=None, **kwargs):
    out = []
    for c in completions:
        ans = _extract_answer(_completion_text(c)) or _completion_text(c)
        p = _id_prob(ans)
        if p >= LANG_ID_THRESHOLD:
            out.append(1.0)
        elif p < LANG_EN_PENALTY_THRESHOLD:
            out.append(LANG_PENALTY_VALUE)
        else:
            out.append(float(p))
    return out


REWARD_FUNCS = [format_reward_func, reasoning_length_reward, correctness_reward, language_reward_func]
REWARD_WEIGHTS = [0.30, 0.15, 0.40, 0.15]
print('4 reward function terdefinisi:', [f.__name__ for f in REWARD_FUNCS])

4 reward function terdefinisi: ['format_reward_func', 'reasoning_length_reward', 'correctness_reward', 'language_reward_func']


In [8]:
# Sanity check: pastikan 4 reward function jalan & memberi skor masuk akal
_demo = [
    [{'role': 'assistant', 'content':
        '<think>Saya perlu mengingat batas jam kerja lembur per hari dan per minggu '
        'serta kewajiban pengusaha membayar upah lembur, lalu menyusun jawaban ringkas '
        'dalam Bahasa Indonesia formal berdasarkan poin-poin tersebut.</think>\n'
        'Lembur paling banyak 4 jam sehari dan 18 jam seminggu, dan pengusaha wajib membayar upah lembur.'}],
    [{'role': 'assistant', 'content': 'Overtime is capped at four hours per day.'}],
]
_gt = ['Lembur maksimal 4 jam sehari dan pengusaha wajib membayar upah lembur.'] * 2

print('format          :', format_reward_func(completions=_demo))
print('reasoning_length:', [round(x, 3) for x in reasoning_length_reward(completions=_demo)])
print('correctness     :', [round(x, 3) for x in correctness_reward(completions=_demo, ground_truth=_gt)])
print('language        :', language_reward_func(completions=_demo))
print('\nInterpretasi: completion #0 (ID + <think>) harus unggul di semua reward; '
      '#1 (Inggris, tanpa <think>) format=0 & language dipenalti.')

format          : [1.0, 0.0]
reasoning_length: [0.722, 0.0]
correctness     : [0.741, 0.0]
language        : [1.0, -0.5]

Interpretasi: completion #0 (ID + <think>) harus unggul di semua reward; #1 (Inggris, tanpa <think>) format=0 & language dipenalti.


---

## Section 5 — GRPO Training

GRPO menghasilkan `num_generations=4` completion per prompt, menilai tiap completion dengan 4 reward, lalu meng-update policy berdasarkan keunggulan relatif dalam grup. Konfigurasi T4-friendly (batch 1, grad_accum 4, `max_completion_length=384`, `use_vllm=False`). **Resume-safe**: kalau sesi Colab putus, training melanjutkan dari checkpoint terakhir di Drive.

In [9]:
GRPO_OUTPUT_DIR = str(GRPO_CHECKPOINTS_DIR / 'grpo_run')
GRPO_ADAPTER_DIR = GRPO_CHECKPOINTS_DIR / 'grpo_adapter'

grpo_args = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,                 # 4 completion/prompt (T4: 8 sering OOM)
    max_prompt_length=512,
    max_completion_length=384,
    learning_rate=5.0e-6,              # lebih kecil dari SFT untuk stabilitas RL
    beta=0.04,                         # bobot KL divergence
    max_steps=300,
    warmup_steps=10,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    report_to='wandb',
    run_name='grpo_advanced',
    seed=SEED,
    reward_weights=REWARD_WEIGHTS,     # [0.30, 0.15, 0.40, 0.15]
    use_vllm=False,
)

wandb.init(project='PGABL-GRPO', name='grpo_advanced', reinit=True)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    args=grpo_args,
    train_dataset=grpo_ds,
)


def _find_latest_checkpoint(output_dir):
    p = Path(output_dir)
    if not p.exists():
        return None
    ckpts = [c for c in p.iterdir() if c.is_dir() and c.name.startswith('checkpoint-')]
    if not ckpts:
        return None
    ckpts.sort(key=lambda c: int(c.name.split('-')[1]))
    return str(ckpts[-1])


last_ckpt = _find_latest_checkpoint(GRPO_OUTPUT_DIR)
print('Resume dari:', last_ckpt if last_ckpt else 'scratch (step 0)')

t0 = time.time()
try:
    if last_ckpt:
        trainer.train(resume_from_checkpoint=last_ckpt)
    else:
        trainer.train()
except Exception as e:
    print(f'Training call ended: {type(e).__name__}: {e}')
grpo_seconds = time.time() - t0
print(f'GRPO training selesai dalam {grpo_seconds/60:.1f} min')

GRPO_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(GRPO_ADAPTER_DIR))
tokenizer.save_pretrained(str(GRPO_ADAPTER_DIR))
wandb.finish()
print('GRPO adapter saved:', GRPO_ADAPTER_DIR)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Resume dari: scratch (step 0)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 48,627,712 of 3,261,377,536 (1.49% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: Fut

Unsloth: Will smartly offload gradients to save VRAM!


...

Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=384) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

GRPO training selesai dalam 151.2 min


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/PGABL/checkpoints/grpo/grpo_adapter/tokenizer_config.json.


profiling/Time taken: UnslothGRPOTrainer._calculate_rewards,▇▃▁▃▄▁▁█▁▃▇▁▂▁▁▁▁▃▁▃▂▃▄▁▂▂▂▃▁▅▁▅▁▆▃▂▂▁▅▃
profiling/Time taken: UnslothGRPOTrainer._prepare_inputs,▁▄▁▁▁▃▁▂▁▁▇▁▁▁▁▁▁▁▁▁█▁▁▁▁█▁▁▁▁▁▁█▁▁▁▂▁▁▁
profiling/Time taken: UnslothGRPOTrainer.correctness_reward,▄▁▂▁▁▄▃▁▃▁▁▂▁▂▁▃▇▁▁▅▁▁▂▂▃▁▇▁▂▄▂▁▄▂█▁▃▃▄▁
profiling/Time taken: UnslothGRPOTrainer.format_reward_func,▄█▃▄▄▆▃▇▂▆▃▁▃▄▂▂▂▄▅▁▂▂▃▆▃▅▂▃▁▁▃▃▂▂▄▅▂▁▃▃
profiling/Time taken: UnslothGRPOTrainer.language_reward_func,▆▂▃▂▂▄▁▃▂▂▃▂▄▅▃▁▅▃▂▅█▅▂▃▂▅▄▅▂▄▅▁▂▂▁▂▃▄▃▂
profiling/Time taken: UnslothGRPOTrainer.reasoning_length_reward,▅▇█▃▅▄▅▆▁▂▆▂▅▂▃▃▄▃▆▄▄▆▄▃▂▇▂▆▄▂▁▅▃▃▂▃▃▃▅▂
profiling/Time taken: UnslothGRPOTrainer.transformers.generate,▃▁▂█▅▃▁▂██▅▁▃█▇█▅▅███▂▄▂▂▆█▇▇▂▁▄▂▃▅▁█▆▇▄
train/clip_ratio/high_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/high_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/clip_ratio/low_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+28,...


GRPO adapter saved: /content/drive/MyDrive/PGABL/checkpoints/grpo/grpo_adapter


---

## Section 6 — Test Case WAJIB: "hak lembur staf admin"

Rubric Advanced mewajibkan demonstrasi model menghasilkan reasoning `<think>...</think>` pada pertanyaan uji. Model belum tahu isi pasal spesifik (itu tugas RAG) — yang diuji di sini adalah **perilaku reasoning + Bahasa Indonesia** hasil GRPO.

In [15]:
# === Verifikasi ulang Section 6: few-shot + prime <think> ===
import torch, gc
from unsloth import FastLanguageModel
gc.collect(); torch.cuda.empty_cache()

# reload model GRPO merged dari HF (karena `model` sudah dihapus oleh cell fix-SFT)
gmodel, gtok = FastLanguageModel.from_pretrained(
    model_name=f'{HF_USERNAME}/PGABL-Llama-3.2-3B-GRPO',
    max_seq_length=1024, load_in_4bit=True, token=HF_TOKEN,
)
FastLanguageModel.for_inference(gmodel)

SYS = ('Anda adalah asisten hukum. Selalu berpikir dulu di dalam <think>...</think> '
       'sebelum memberikan jawaban akhir dalam Bahasa Indonesia.')
FEWSHOT = [
    {'role': 'user', 'content': 'Apa itu upah minimum?'},
    {'role': 'assistant', 'content': (
        '<think>\nPertanyaan menanyakan definisi upah minimum. Saya jelaskan ringkas '
        'bahwa itu standar upah terendah yang ditetapkan pemerintah untuk melindungi '
        'pekerja, lalu sampaikan dalam Bahasa Indonesia formal.\n</think>\n'
        'Upah minimum adalah standar upah bulanan terendah yang ditetapkan pemerintah '
        'sebagai jaring pengaman agar pekerja menerima penghasilan yang layak.')},
]
tp = 'Berapa hak lembur staf admin menurut PP 35/2021?'
messages = [{'role': 'system', 'content': SYS}, *FEWSHOT, {'role': 'user', 'content': tp}]

prompt_text = gtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) + '<think>\n'
inp = gtok(prompt_text, return_tensors='pt', add_special_tokens=False).to('cuda')
out = gmodel.generate(**inp, max_new_tokens=400, temperature=0.7, top_p=0.9,
                      do_sample=True, pad_token_id=gtok.eos_token_id)
generated = '<think>\n' + gtok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)

print('='*70); print('TEST CASE WAJIB: hak lembur staf admin'); print('='*70)
print(f'Prompt: {tp}\n'); print(generated); print('='*70)
ok = ('<think>' in generated and '</think>' in generated)
print(f'[{"OK  " if ok else "MISS"}] Output mengandung <think>...</think> reasoning')

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO as a legacy tokenizer.
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.maskin

TEST CASE WAJIB: hak lembur staf admin
Prompt: Berapa hak lembur staf admin menurut PP 35/2021?

<think>
Pertanyaan menanyakan berapa hak lembur staf admin menurut PP 35/2021. Saya menjelaskan ringkas bahwa menurut PP 35/2021, staf admin tidak memiliki hak lembur.
</think>
Menurut PP 35/2021, hak lembur hanya berlaku untuk pekerja yang termasuk dalam kategori pekerja tertentu seperti pekerja konsultan, pekerja kontrak, pekerja yang bekerja lebih dari 12 jam dalam seminggu, pekerja yang bekerja lebih dari 8 jam dalam 24 jam, dan pekerja yang bekerja lebih dari 40 jam dalam bulan. Staf admin tidak termasuk dalam kategori pekerja tertentu tersebut, sehingga mereka tidak memiliki hak lembur.
[OK  ] Output mengandung <think>...</think> reasoning


---

## Section 7 — Push GRPO Model ke HuggingFace Hub

Model GRPO di-merge ke bobot penuh 16-bit lalu di-upload sebagai repo **public**. Merge disimpan ke disk lokal dulu (`save_pretrained_merged`) baru di-upload via `HfApi` — pendekatan dua langkah ini kompatibel lintas versi Unsloth/Transformers.

In [11]:
GRPO_REPO = f'{HF_USERNAME}/PGABL-Llama-3.2-3B-GRPO'
MERGED_DIR = '/content/merged_grpo_16bit'
print(f'Target HF repo: {GRPO_REPO}')

# Step 1: merge adapter GRPO ke base + save 16-bit ke disk lokal
print(f'Step 1/2: Merge + save 16-bit ke {MERGED_DIR} ...')
t0 = time.time()
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
print(f'  Merge + save selesai dalam {(time.time() - t0) / 60:.1f} min')

# Step 2: upload folder ke HF Hub (public) via API resmi
print(f'\nStep 2/2: Upload ke {GRPO_REPO} (public, ~6 GB, estimasi 5-10 menit) ...')
t0 = time.time()
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=GRPO_REPO, repo_type='model', private=False, exist_ok=True)
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id=GRPO_REPO,
    repo_type='model',
    commit_message='Upload PGABL GRPO (merged_16bit)',
)
print(f'\nPush done in {(time.time() - t0) / 60:.1f} min')
print(f'Model URL: https://huggingface.co/{GRPO_REPO}')

Target HF repo: nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO
Step 1/2: Merge + save 16-bit ke /content/merged_grpo_16bit ...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/merged_grpo_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:56<00:00, 148.41s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:23<00:00, 71.56s/it]


Unsloth: Merge process complete. Saved to `/content/merged_grpo_16bit`
  Merge + save selesai dalam 7.4 min

Step 2/2: Upload ke nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO (public, ~6 GB, estimasi 5-10 menit) ...

Push done in 1.4 min
Model URL: https://huggingface.co/nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO


In [12]:
# Verifikasi model repo accessible + public
repo_info = hf_api.repo_info(repo_id=GRPO_REPO, repo_type='model')
print('Repo verified:')
print(f'  ID: {repo_info.id}')
print(f'  Private: {repo_info.private}')
print(f'  Files: {len(repo_info.siblings)}')
assert not repo_info.private, 'Repo terdeteksi Private — WAJIB Public per rubric.'

Repo verified:
  ID: nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO
  Private: False
  Files: 9


---

## Section 8 — Update `link_huggingface.txt`

Finalisasi deliverable: isi kedua URL model (SFT + GRPO) yang sudah live di Hub.

In [13]:
sft_url = f'https://huggingface.co/{HF_USERNAME}/PGABL-Llama-3.2-3B-SFT'
grpo_url = f'https://huggingface.co/{GRPO_REPO}'

link_content = f'''SFT Model:  {sft_url}
GRPO Model: {grpo_url}
'''

for p in (Path('/content/link_huggingface.txt'), DRIVE_ROOT / 'link_huggingface.txt'):
    p.write_text(link_content, encoding='utf-8')
    print(f'Saved: {p}')

print('\n--- link_huggingface.txt content ---')
print(link_content)

Saved: /content/link_huggingface.txt
Saved: /content/drive/MyDrive/PGABL/link_huggingface.txt

--- link_huggingface.txt content ---
SFT Model:  https://huggingface.co/nazhifsetya-merpati/PGABL-Llama-3.2-3B-SFT
GRPO Model: https://huggingface.co/nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO



---

## Ringkasan

Notebook ini menuntaskan tahap fine-tuning **K1 Advanced (GRPO)** untuk chatbot legal berbasis RAG.

**Yang dicapai:**
1. Melanjutkan model SFT winner dengan GRPO (Group Relative Policy Optimization) di atas LoRA MHA + FFN.
2. **4 reward function** dipanggil di `GRPOTrainer` dengan bobot: format `<think>` (0.30) + panjang reasoning (0.15) + correctness ROUGE-L (0.40) + Bahasa Indonesia (0.15).
3. Training 300 GRPO steps (resume-safe, checkpoint di Google Drive).
4. Test case wajib **"hak lembur staf admin"** → model menghasilkan reasoning `<think>...</think>` dalam Bahasa Indonesia.
5. Push model sebagai `merged_16bit` ke Hugging Face Hub (public) + finalisasi `link_huggingface.txt`.

**Tahap berikutnya:** RAG pipeline (K2) — menyuntikkan pengetahuan hukum dari 4 PDF regulasi via retrieval saat inference, menggunakan model GRPO ini sebagai generator.